# P02 — Auditoría de fairness con Fairlearn

En P01 evaluamos el rendimiento global del modelo de Velora Finance. Ahora auditamos si ese comportamiento es comparable entre grupos.

`customer_group` se utiliza **solo para auditoría**: no forma parte de las variables predictoras.

El notebook utiliza **Fairlearn 0.15.0**, una librería específica para evaluación y mitigación de fairness. El código está preparado; la interacción principal consiste en cambiar el `threshold` y comparar métricas.


## 1. Instalar Fairlearn

La primera celda instala una versión concreta para que el comportamiento sea reproducible.


In [ ]:
!pip -q install fairlearn==0.15.0


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

GENERATOR_URL = "https://raw.githubusercontent.com/pacosepulveda/ia-confiable-explicable-labs/main/resources/generate_velora_credit.py"
namespace = {"__name__": "velora_generator"}
with urllib.request.urlopen(GENERATOR_URL) as response:
    exec(response.read().decode("utf-8"), namespace)

df = namespace["generate_dataset"]()
print("Dataset:", df.shape)

from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    demographic_parity_difference,
    demographic_parity_ratio,
    equal_opportunity_difference,
    equalized_odds_difference,
    false_positive_rate,
    false_negative_rate,
)
print("Fairlearn preparado.")


## 2. Recrear exactamente el modelo de P01

Usamos el mismo dataset, variables, partición train/test y regresión logística. Así cualquier diferencia observada proviene del análisis por grupos, no de haber cambiado el modelo.


In [ ]:
FEATURES = [
    "age", "annual_income", "debt_ratio", "employment_months",
    "late_payments_12m", "credit_history_months", "existing_debt",
    "requested_amount", "savings_balance",
]
X = df[FEATURES]
y = df["default"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

audit = df.loc[X_test.index, ["application_id", "customer_group", "default"]].copy()
audit["prob_default"] = proba
print("Modelo de P01 recreado.")
print("customer_group es predictor:", "customer_group" in FEATURES)


## 3. Contexto antes de medir fairness

Antes de comparar predicciones observamos cuántos casos hay en cada grupo y sus tasas reales de impago. Las tasas base importan porque distintos criterios de fairness formalizan preguntas diferentes.


In [ ]:
context = audit.groupby("customer_group").agg(
    n=("default", "size"),
    tasa_real_impago=("default", "mean"),
    riesgo_medio_predicho=("prob_default", "mean"),
)
display(context.style.format({
    "tasa_real_impago": "{:.2%}",
    "riesgo_medio_predicho": "{:.2%}",
}))


## 4. Auditoría con MetricFrame

Para Fairlearn definiremos el **resultado favorable** como `solvente/aprobado = 1`. Esto evita confundir el significado de la clase positiva: en P01 `default=1` era el evento adverso.

Con un threshold dado:

- `solvente_real = 1 - default`
- `aprobado = 1 - pred_default`


In [ ]:
def audit_threshold(threshold):
    out = audit.copy()
    out["pred_default"] = (out["prob_default"] >= threshold).astype(int)
    out["solvente_real"] = 1 - out["default"]
    out["aprobado"] = 1 - out["pred_default"]

    metrics = {
        "accuracy": accuracy_score,
        "tasa_aprobacion": selection_rate,
        "recall_solventes": recall_score,
        "precision_aprobacion": precision_score,
        "FPR_aprobacion": false_positive_rate,
        "FNR_aprobacion": false_negative_rate,
    }

    frame = MetricFrame(
        metrics=metrics,
        y_true=out["solvente_real"],
        y_pred=out["aprobado"],
        sensitive_features=out["customer_group"],
    )

    summary = pd.Series({
        "demographic_parity_difference": demographic_parity_difference(
            out["solvente_real"], out["aprobado"],
            sensitive_features=out["customer_group"]
        ),
        "demographic_parity_ratio": demographic_parity_ratio(
            out["solvente_real"], out["aprobado"],
            sensitive_features=out["customer_group"]
        ),
        "equal_opportunity_difference": equal_opportunity_difference(
            out["solvente_real"], out["aprobado"],
            sensitive_features=out["customer_group"]
        ),
        "equalized_odds_difference": equalized_odds_difference(
            out["solvente_real"], out["aprobado"],
            sensitive_features=out["customer_group"]
        ),
    })
    return out, frame, summary

_, frame_050, fairness_050 = audit_threshold(0.50)
print("Métricas globales")
display(frame_050.overall.to_frame("valor").style.format("{:.2%}"))
print("Métricas por grupo")
display(frame_050.by_group.style.format("{:.2%}"))
print("Diferencias/ratios de fairness")
display(fairness_050.to_frame("valor").style.format("{:.3f}"))


### Cómo interpretar el resultado

No buscamos que todos los números sean idénticos. Buscamos detectar **qué dimensión cambia entre grupos**.

Una tasa de aprobación relativamente parecida puede coexistir con diferencias importantes en tasas de error. Eso demuestra por qué una auditoría no debería reducir fairness a una única métrica.


## 5. Visualizar tasas de error por grupo


In [ ]:
plot_table = frame_050.by_group[["tasa_aprobacion", "FPR_aprobacion", "FNR_aprobacion"]]
plot_table.plot(kind="bar", figsize=(8, 5))
plt.ylabel("Tasa")
plt.title("Métricas por grupo — threshold 0.50")
plt.xticks(rotation=0)
plt.show()


## 6. Cambiar el threshold

El modelo no cambia. Cambia la política que transforma el score en decisión. Observa cómo una decisión operativa puede modificar simultáneamente rendimiento y fairness.


In [ ]:
THRESHOLD_ELEGIDO = 0.50 #@param {type:"slider", min:0.10, max:0.80, step:0.05}
_, selected_frame, selected_fairness = audit_threshold(THRESHOLD_ELEGIDO)
print(f"Threshold: {THRESHOLD_ELEGIDO:.2f}")
print()
print("Por grupo")
display(selected_frame.by_group.style.format("{:.2%}"))
print()
print("Fairness")
display(selected_fairness.to_frame("valor").style.format("{:.3f}"))


### Discusión

- ¿Hay un threshold que haga desaparecer todas las diferencias?
- ¿Mejorar demographic parity mejora necesariamente equalized odds?
- ¿Qué tipo de error importa más en este caso?
- ¿Qué criterio sería defendible ante riesgo, negocio y auditoría?

**No existe un criterio de fairness universalmente correcto.** La métrica debe corresponder al daño y al contexto que queremos evaluar.


## 7. No usar el atributo sensible no elimina la necesidad de auditar

`customer_group` no participa en el entrenamiento, pero Fairlearn puede mostrar diferencias entre grupos. Las causas pueden estar en distribuciones distintas, variables proxy, calidad de datos, etiquetas históricas o decisiones previas del proceso.


In [ ]:
print("Variables utilizadas por el modelo:")
print(FEATURES)
print()
print("customer_group incluido:", "customer_group" in FEATURES)


## 8. Profundización — intersecciones

En una auditoría real rara vez basta con comparar un único atributo. `MetricFrame` también permite trabajar con varias características sensibles y analizar intersecciones.

En este dataset no añadimos atributos artificiales solo para producir más combinaciones. La idea queda preparada para datasets reales posteriores.


# Conclusiones de P02

- Las métricas globales pueden ocultar diferencias entre grupos.
- Eliminar una variable sensible del entrenamiento no demuestra fairness.
- Fairlearn permite desagregar métricas y calcular criterios formales con una metodología reproducible.
- Demographic parity, equal opportunity y equalized odds responden a preguntas diferentes.
- El threshold forma parte de la auditoría porque modifica la distribución de errores.
- Una disparidad estadística es una señal que debe investigarse; no es por sí sola una conclusión ética, causal o jurídica.
